In [1]:
from datasets import load_dataset, Dataset
from tqdm import tqdm
import numpy as np
import os
# Force PyArrow to use standard C system memory allocation
#os.environ["ARROW_DEFAULT_MEMORY_POOL"] = "system"
import pyarrow.parquet as pq
import pyarrow as pa
import json


In [2]:
df = pq.ParquetFile('../data/ds_select_size.parquet')


In [3]:
df.metadata

  created_by: parquet-cpp-arrow version 25.0.1
  num_columns: 4
  num_rows: 34133922
  num_row_groups: 1847
  format_version: 2.6
  serialized_size: 1198529

In [ ]:
# data leak tqdm
count_words = 0
with open('../data/count_words.ndjson', 'w') as f:
    for batch in df.iter_batches(100_000, columns=["text"]): #, use_threads=False
        for text in batch['text']:
            l = len(str(text).split())
            count_words += l
            f.write(json.dumps({'count_words': l}) + '\n')
            f.flush()         
        #pa.default_memory_pool().release_unused()

    df.close()

print("Total: ", count_words)
print("Average: ", count_words/df.metadata.num_rows)

In [ ]:
ds = Dataset.from_parquet('../data/ds_select_size.parquet', split='train', streaming=True)


Generating train split: 0 examples [00:00, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

In [ ]:
ds.push_to_hub("unb-labia/GigaVerbo_filtered_size_ge1024times1.2_tkfert", private=True, token=os.environ.get("HUGGINGFACE_TOKEN"))
